# 🧑‍🍳 Neosantara AI Cookbook: Jev Decision Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neosantara-xyz/examples/blob/main/cookbook/advanced/jev-decision-models.ipynb)

**Jev** from TypeSafe is a *System One* model. It does not write text. You give it some data and a few closed questions, and it returns a probability for every possible answer. Your code then decides what to do.

This notebook triages Indonesian marketplace support tickets with plain HTTP calls, no framework needed.

| Question type | What you ask | What Jev returns |
| :--- | :--- | :--- |
| `noul` | A yes/no question | `noul`: the probability the answer is yes |
| `score` | A question with ordered levels | `score` (expected level) and `probabilities` per level |
| `choice` | Pick one of several options | `choice` (most likely option) and `probabilities` per option |

Jev is not available on the Free tier. Input costs $0.042 per 1M tokens and output tokens are free.

### 📖 Documentation
- [Jev-style decision programs with DSPy](https://docs.neosantara.xyz/en/guides/dspy-decision-programs)
- [Model catalog](https://docs.neosantara.xyz/en/gateway/models)

## Setup

Installs `requests` and loads your Neosantara API key from Colab Secrets, or asks you to paste it.

**New to Neosantara?** Sign up and create an API key at [app.neosantara.xyz/api-keys](https://app.neosantara.xyz/api-keys).

In [ ]:
!pip install -q requests

import os
from getpass import getpass

try:
    from google.colab import userdata
    NEOSANTARA_API_KEY = userdata.get("NEOSANTARA_API_KEY")
    if not NEOSANTARA_API_KEY:
        raise ValueError("Secret not found")
    os.environ["NEOSANTARA_API_KEY"] = NEOSANTARA_API_KEY
    print("✅ API Key loaded from Google Colab Secrets!")
except (ImportError, Exception):
    if "NEOSANTARA_API_KEY" not in os.environ:
        print("🔑 API Key not found in Secrets. Please paste it below:")
        print("   No key yet? Create one at https://app.neosantara.xyz/api-keys")
        os.environ["NEOSANTARA_API_KEY"] = getpass("Neosantara API Key: ")
    else:
        print("✅ API Key loaded from environment!")

## Define the questions

Every question has three parts:

- `type`: `noul`, `score`, or `choice`.
- `instructions`: the question itself, in plain language.
- `criteria`: what each answer means. For `noul` it describes `true` and `false`. For `score` it is a list of levels from lowest to highest. For `choice` it maps each option name to a description.

Jev never sees the question names (`escalate`, `severity`, `team`). They are only keys for reading the answers back.

In [ ]:
QUESTIONS = {
    "escalate": {
        "type": "noul",
        "instructions": "Apakah tiket ini perlu ditangani agen manusia?",
        "criteria": {"true": "Butuh agen manusia", "false": "Bisa dibalas otomatis"},
    },
    "severity": {
        "type": "score",
        "instructions": "Seberapa besar dampaknya ke pelanggan?",
        "criteria": ["Pertanyaan umum", "Transaksi terganggu", "Uang pelanggan tertahan"],
    },
    "team": {
        "type": "choice",
        "instructions": "Tim mana yang harus menangani tiket ini?",
        "criteria": {
            "pembayaran": "QRIS, virtual account, e-wallet, refund",
            "pengiriman": "Resi, kurir, paket telat atau hilang",
            "akun": "Login, OTP, verifikasi KYC",
            "teknis": "Aplikasi error, crash, fitur tidak jalan",
        },
    },
}

## Send the request

`ask_jev()` sends one `POST /v1/systemone` request with:

| Field | Value |
| :--- | :--- |
| `model` | `"jev"` |
| `state` | The data to judge. Here it is the ticket text; it can also be a JSON object |
| `questions` | The `QUESTIONS` dictionary above |

`res.raise_for_status()` stops with a clear error if the request fails, for example with a wrong API key or a Free-tier account. The response has `answers` (one entry per question) and `usage` (tokens billed).

In [ ]:
import json
import requests

BASE_URL = "https://api.neosantara.xyz"
HEADERS = {
    "Authorization": f"Bearer {os.environ['NEOSANTARA_API_KEY']}",
    "Content-Type": "application/json",
}


def ask_jev(ticket: str) -> dict:
    res = requests.post(
        f"{BASE_URL}/v1/systemone",
        headers=HEADERS,
        json={"model": "jev", "state": ticket, "questions": QUESTIONS},
        timeout=30,
    )
    res.raise_for_status()
    return res.json()


result = ask_jev("Min, saldo udah kepotong pas bayar QRIS tapi status order masih menunggu pembayaran.")
print(json.dumps(result, indent=2, ensure_ascii=False))

## Turn probabilities into actions

Jev only reports how likely each answer is. `decide()` applies three rules you control:

| Setting | Rule | Why |
| :--- | :--- | :--- |
| `ESCALATE_THRESHOLD = 0.3` | Escalate when P(yes) is at least 0.3 | Lower than 0.5 on purpose: a missed urgent ticket costs more than an extra escalation |
| `SEVERITY_CUTS = [0.5, 1.5]` | `score` below 0.5 is level 0, below 1.5 is level 1, otherwise level 2 | Turns the expected score into a whole level |
| `TEAM_WEIGHTS = {"pembayaran": 1.2}` | Multiply the payments probability by 1.2 before picking the highest | Money tickets win close calls |

Changing these numbers never requires a new request, so you can tune them freely.

In [ ]:
ESCALATE_THRESHOLD = 0.3
SEVERITY_CUTS = [0.5, 1.5]
TEAM_WEIGHTS = {"pembayaran": 1.2}


def decide(answers: dict) -> dict:
    p_escalate = answers["escalate"]["noul"]
    severity_level = sum(1 for cut in SEVERITY_CUTS if answers["severity"]["score"] >= cut)
    team_probs = answers["team"]["probabilities"]
    team = max(team_probs, key=lambda k: team_probs[k] * TEAM_WEIGHTS.get(k, 1.0))
    return {
        "action": "antrian_agen" if p_escalate >= ESCALATE_THRESHOLD else "balas_otomatis",
        "p_escalate": p_escalate,
        "severity": severity_level,
        "team": team,
    }


for ticket in [
    "Min, saldo udah kepotong pas bayar QRIS tapi status order masih menunggu pembayaran.",
    "Kak, resi JNE saya kok belum update dari kemarin?",
    "Kode OTP ga masuk-masuk, udah coba 3 kali.",
]:
    print(ticket)
    print("  ->", decide(ask_jev(ticket)["answers"]))

## Try your own tickets

Type a customer message and see how Jev routes it. Press Enter on an empty line to stop.

In [ ]:
while True:
    ticket = input("Tiket pelanggan (kosongkan untuk berhenti): ").strip()
    if not ticket:
        break
    answers = ask_jev(ticket)["answers"]
    decision = decide(answers)
    print(f"  Tim        : {decision['team']}  {answers['team']['probabilities']}")
    print(f"  Severity   : {decision['severity']} (score {answers['severity']['score']})")
    print(f"  P(eskalasi): {decision['p_escalate']}")
    print(f"  Aksi       : {decision['action']}")
    print()

## Vercel AI Gateway format

`POST /v1/evaluate` accepts the same questions in the Vercel AI Gateway format. The only difference for yes/no questions: send `"type": "boolean"` and read `probability`.

In [ ]:
res = requests.post(
    f"{BASE_URL}/v1/evaluate",
    headers=HEADERS,
    json={
        "model": "jev",
        "state": "Kode OTP ga masuk-masuk, udah coba 3 kali.",
        "questions": {
            "escalate": {"type": "boolean", "instructions": "Apakah tiket ini perlu ditangani agen manusia?"}
        },
    },
    timeout=30,
)
print(res.status_code, json.dumps(res.json(), indent=2, ensure_ascii=False))

## Next steps

| Task | Link |
| :--- | :--- |
| Declare the same decisions as DSPy signatures and let `ReAnchor` tune the thresholds | [DSPy Decision Types](https://colab.research.google.com/github/neosantara-xyz/examples/blob/main/cookbook/advanced/dspy-decision-types.ipynb) |
| Read the full guide | [Jev-style decision programs](https://docs.neosantara.xyz/en/guides/dspy-decision-programs) |